# Phase 3: label complaints with an open teacher model (Kaggle, free GPU)

Runs `label/teacher.py` against a vLLM server on Kaggle's free GPUs. Nothing is paid.

**Before running**
1. Settings: *Accelerator* = **GPU T4 x2**, *Internet* = **on**.
2. Upload `data/processed/train.parquet` and `val.parquet` as a private Kaggle dataset and add it
   to this notebook (*Add Input*). Set `SPLITS_DIR` below to its path.
3. To resume an earlier session: download `labels/*.jsonl` from its output, upload them as a
   dataset, add it, and set `PREVIOUS_LABELS`. Rows already labelled by the same model and
   prompt version are skipped.
4. The repo on GitHub must contain the `label/` folder (push before running).

Label **val first** (5k rows), score it against the gold set on the laptop, and only then run train.

In [ ]:
MODEL = "Qwen/Qwen2.5-14B-Instruct-AWQ"   # fallback if it does not fit or is slow: Qwen/Qwen2.5-7B-Instruct-AWQ
TENSOR_PARALLEL = 2                         # T4 x2; use 1 for a single GPU
SPLIT = "val"                               # then "train"
SPLITS_DIR = "/kaggle/input/complaint-splits"
PREVIOUS_LABELS = None                      # e.g. "/kaggle/input/complaint-labels"
REPO = "https://github.com/bhzcode1/customer-s-complaint.git"

In [ ]:
!git clone -q {REPO} /kaggle/working/repo
!pip install -q vllm duckdb

In [ ]:
import os, shutil, glob
os.chdir("/kaggle/working/repo")
os.makedirs("data/processed", exist_ok=True)
os.makedirs("data/labels", exist_ok=True)
for f in glob.glob(f"{SPLITS_DIR}/*.parquet"):
    shutil.copy(f, "data/processed/")
if PREVIOUS_LABELS:
    for f in glob.glob(f"{PREVIOUS_LABELS}/*.jsonl"):
        shutil.copy(f, "data/labels/")
print(os.listdir("data/processed"), os.listdir("data/labels"))

In [ ]:
# Start vLLM in the background. T4 has no bfloat16, hence --dtype half.
import subprocess, time, urllib.request
log = open("/kaggle/working/vllm.log", "w")
server = subprocess.Popen(
    ["vllm", "serve", MODEL, "--dtype", "half", "--max-model-len", "4096",
     "--tensor-parallel-size", str(TENSOR_PARALLEL), "--gpu-memory-utilization", "0.92",
     "--served-model-name", MODEL, "--port", "8000"],
    stdout=log, stderr=subprocess.STDOUT)
for _ in range(120):  # model download + load can take ~10 min
    try:
        urllib.request.urlopen("http://localhost:8000/health", timeout=5); print("server ready"); break
    except Exception:
        time.sleep(10)
else:
    print(open("/kaggle/working/vllm.log").read()[-3000:]); raise RuntimeError("vLLM did not start")

In [ ]:
# Smoke test: 20 rows into a separate file, look at them before the full run.
!python label/teacher.py --split {SPLIT} --limit 20 --workers 16 --model {MODEL}     --base-url http://localhost:8000/v1 --out data/labels/smoke_{SPLIT}.jsonl
!head -c 2000 data/labels/smoke_{SPLIT}.jsonl

In [ ]:
# Full run. Prints rows/s every 50 rows; safe to stop and resume (see step 3 above).
!python label/teacher.py --split {SPLIT} --workers 32 --model {MODEL} --base-url http://localhost:8000/v1

In [ ]:
# Copy results to the notebook output so they can be downloaded (Output tab, or "Save Version").
os.makedirs("/kaggle/working/labels", exist_ok=True)
for f in glob.glob("data/labels/*.jsonl"):
    shutil.copy(f, "/kaggle/working/labels/")
print(os.listdir("/kaggle/working/labels"))
server.terminate()